# Analytics Run Evidence (Issue #60)

Confirms that the BQ1 and BQ2 Analytics tables in `05_analytics` are correct, consistent with Gold, and safe to use for downstream analysis.

This notebook is **read-only**. It reads the Analytics tables, `05_analytics.analytics_run_log` and `06_data_quality.dq_results`. It does not write any table.

**Already checked by the Analytics notebooks (not repeated here):**
- `01_build_analytics`: Gold input checks, per-table BLOCK checks before each table is written, and cross-table consistency
- `02_analytics_validation`: lineage, natural keys, columns, row counts, recomputation of every derived metric, and reconciliation of base values to Gold at the recorded Gold versions

Both save their results to `dq_results` and `analytics_run_log`. Section 1 **reports** those results; it does not recalculate them.

**Added in this notebook:**
- **Section 2:** selected units compared against values calculated directly from the **source files** (DOT PDFs, PSA tables, DS_18), which neither Analytics notebook does
- **Section 3:** exclusions applied, segment coverage and caveats, counted by reason

| Section | What it shows |
| --- | --- |
| 1 | Latest build and validation runs, their checks, and the lineage of the tables |
| 2 | Edge cases: Analytics values vs values calculated from the source files |
| 3 | Exclusions, segment coverage and caveats |

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG, ANALYTICS, DQ = "tuklas_dev", "05_analytics", "06_data_quality"
spark.sql(f"USE CATALOG {CATALOG}")
def t(schema, name): return f"`{schema}`.`{name}`"

BUILD_NB, VALIDATION_NB = "01_build_analytics", "02_analytics_validation"
TOL = 0.01   # tolerance for % and per-1,000 values (Gold stores money rounded to whole thousand PhP)

## 1. Latest runs, checks and lineage

Shows the latest run of each Analytics notebook from `analytics_run_log`, then every check those two runs saved to `dq_results`, failures first. In `dq_results`, a REVIEW check with status `FAIL` is a finding to review, not a failure.

It also confirms that the persisted tables come from the latest build run and that validation ran after it.

In [0]:
log = spark.table(t(ANALYTICS, "analytics_run_log"))
w = Window.partitionBy("notebook_name").orderBy(F.col("started_at").desc())
latest = log.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
display(latest.orderBy("notebook_name"))

build = latest.filter(F.col("notebook_name") == BUILD_NB).first()
val = latest.filter(F.col("notebook_name") == VALIDATION_NB).first()
print(f"Build run:      {build['run_id']} ({build['status']})")
print(f"Validation run: {val['run_id']} ({val['status']})")

# Lineage: the persisted tables should all come from the latest build run
lineage = None
for tbl in ["tourism_activity", "tourism_concentration", "tourism_recovery", "traveler_segment_recovery", "tourism_economy"]:
    part = (spark.table(t(ANALYTICS, tbl))
            .select(F.lit(tbl).alias("table_name"), "_analytics_run_id", "_gold_fact_version",
                    "_gold_dim_province_version", "_gold_dim_year_version", "_processed_at").distinct())
    lineage = part if lineage is None else lineage.unionByName(part)
display(lineage.withColumn("from_latest_build", F.col("_analytics_run_id") == F.lit(build["run_id"])))
print("Validation ran after the build:", val["started_at"] >= build["completed_at"])

dq = spark.table(t(DQ, "dq_results")).filter(F.col("run_id").isin(build["run_id"], val["run_id"]))
display(dq.groupBy("notebook_name", "severity", "status").count().orderBy("notebook_name", "severity", "status"))
display(dq.select("notebook_name", "check_id", "table_name", "severity", "status", "observed")
          .orderBy(F.col("status").desc(), "notebook_name", "check_id"))

## 2. Edge cases: Analytics vs source files

Expected values were calculated directly from DOT *Regional Distribution of Overnight Travelers* (2019, 2022, 2023), PSA Tables 3.72, 3.94 and 3.95, and PSA DS_18, applying DL-011 (Lucena in Quezon), DL-013 (Boracay/Aklan), DL-014 (Clark/Pampanga), DL-019 (`-` values) and DL-021 (population).

| Unit | Edge case |
| --- | --- |
| Aklan | Boracay; not comparable to 2019 from 2021 |
| Pampanga | Clark Freeport Zone added back from 2023 |
| Quezon | Lucena inside Quezon |
| City of Baguio | Separately reported HUC |
| Abra | Missing DOT value (2023 `not_reported`) |
| Basilan | No DOT data |
| Aurora, Camiguin | Units shown in the presentation |

Comparison tables are read at `baseline_year = 2019`. Traveler and population counts must match exactly; % and per-1,000 values within 0.01. NULL on both sides counts as a match. A&F growth is checked against constant prices and A&F share against current prices, so a match also confirms the price basis.

In [0]:
# (code, unit, trav_2019, trav_2022, trav_2023, pop_2023, intensity_2023, RI_2022, RI_2023,
#  tourism_change_pct, afs_gva_growth_pct, afs_share_of_gdp_pct, pattern)
EXPECTED = [
    ("0600400000", "Aklan",          2251966, 1759592, 2200138,  620744, 3544.36, None,   None,   None,   -35.05,  3.24, None),
    ("0305400000", "Pampanga",        910666, 1124190, 1655887, 2465728,  671.56, 123.45, 181.83,  81.83,   6.50,  2.65, "tourism_up_economy_up"),
    ("0405600000", "Quezon",         1301486, 1186369, 1264181, 2328096,  543.01,  91.15,  97.13,  -2.87,  50.57,  4.05, "tourism_down_economy_up"),
    ("1430300000", "City of Baguio", 1536458, 1042309, 1309930,  371374, 3527.25,  67.84,  85.26, -14.74,   9.35,  9.96, "tourism_down_economy_up"),
    ("1400100000", "Abra",              2948,    1511,    None,  250399,    None,  51.26,   None,   None,  57.16,  0.61, None),
    ("1900700000", "Basilan",           None,    None,    None,  413710,    None,   None,   None,   None,   0.84,  0.10, None),
    ("0307700000", "Aurora",          273906,  515739,  736339,  234084, 3145.62, 188.29, 268.83, 168.83, -25.05,  1.27, "tourism_up_economy_down"),
    ("1001800000", "Camiguin",        804745,  127859,  259010,   94373, 2744.53,  15.89,  32.19, -67.81, -15.35, 11.80, "tourism_down_economy_down"),
]
exp = spark.createDataFrame(EXPECTED, schema=(
    "province_psgc STRING, unit_name STRING, travelers_2019 BIGINT, travelers_2022 BIGINT, travelers_2023 BIGINT, "
    "population_2023 BIGINT, intensity_2023 DOUBLE, recovery_index_2022 DOUBLE, recovery_index_2023 DOUBLE, "
    "tourism_change_pct DOUBLE, afs_gva_growth_pct DOUBLE, afs_share_of_gdp_pct DOUBLE, tourism_economy_pattern STRING"))

act_activity = (spark.table(t(ANALYTICS, "tourism_activity")).filter("year = 2023")
                .select("province_psgc",
                        F.col("population_total").alias("a_population_2023"),
                        F.col("tourism_intensity_per_1000").alias("a_intensity_2023")))
act_recovery = (spark.table(t(ANALYTICS, "tourism_recovery")).filter("baseline_year = 2019")
                .groupBy("province_psgc").pivot("comparison_year", [2022, 2023])
                .agg(F.first("travelers_comparison_year").alias("trav"),
                     F.first("recovery_index").alias("ri"),
                     F.first("travelers_2019").alias("t19")))
act_economy = (spark.table(t(ANALYTICS, "tourism_economy")).filter("baseline_year = 2019 AND comparison_year = 2023")
               .select("province_psgc",
                       F.col("tourism_change_pct").alias("a_tourism_change_pct"),
                       F.col("afs_gva_growth_pct").alias("a_afs_gva_growth_pct"),
                       F.col("afs_share_of_gdp_pct").alias("a_afs_share_of_gdp_pct"),
                       F.col("tourism_economy_pattern").alias("a_pattern")))

cmp = (exp.join(act_activity, "province_psgc", "left")
          .join(act_recovery, "province_psgc", "left")
          .join(act_economy, "province_psgc", "left"))

# expected column -> actual column
pairs = [("travelers_2019", "2022_t19"), ("travelers_2022", "2022_trav"), ("travelers_2023", "2023_trav"),
         ("population_2023", "a_population_2023"), ("intensity_2023", "a_intensity_2023"),
         ("recovery_index_2022", "2022_ri"), ("recovery_index_2023", "2023_ri"),
         ("tourism_change_pct", "a_tourism_change_pct"), ("afs_gva_growth_pct", "a_afs_gva_growth_pct"),
         ("afs_share_of_gdp_pct", "a_afs_share_of_gdp_pct")]

long = None
for e, a in pairs:
    part = cmp.select("unit_name", F.lit(e).alias("metric"),
                      F.col(e).cast("double").alias("expected"),
                      F.col(f"`{a}`").cast("double").alias("actual"))
    long = part if long is None else long.unionByName(part)

# Counts must match exactly; % and per-1,000 values within TOL
long = (long
    .withColumn("diff", F.when(F.col("expected").isNull() & F.col("actual").isNull(), F.lit(0.0))
                         .otherwise(F.abs(F.col("expected") - F.col("actual"))))
    .withColumn("limit", F.when(F.col("metric").startswith("travelers") | (F.col("metric") == "population_2023"), F.lit(0.0))
                          .otherwise(F.lit(TOL)))
    .withColumn("ok", F.col("diff").isNotNull() & (F.col("diff") <= F.col("limit")))
    .drop("limit"))

display(long.orderBy("ok", "unit_name", "metric"))
display(cmp.select("unit_name", "tourism_economy_pattern", "a_pattern",
                   F.col("tourism_economy_pattern").eqNullSafe(F.col("a_pattern")).alias("ok")))
print(f"Edge-case values not matching: {long.filter('NOT ok').count()} of {long.count()}")

## 3. Exclusions, segment coverage and caveats

Units stay in the Analytics tables even when a metric does not apply, with the reason recorded. This section counts those exclusions by reason, shows the recovery classes, tourism–economy patterns and destination labels, the traveler-segment coverage for the 2023 below-2019 cohort, and the Gold caveat fields carried into Analytics.

In [0]:
rec = spark.table(t(ANALYTICS, "tourism_recovery")).filter("baseline_year = 2019")
display(rec.filter("NOT recovery_eligible").groupBy("comparison_year", "recovery_exclusion_reason").count()
           .orderBy("comparison_year", "recovery_exclusion_reason"))
display(rec.groupBy("comparison_year", "recovery_class").count().orderBy("comparison_year", "recovery_class"))
display(rec.filter("comparison_year = 2023").groupBy("destination_label").count().orderBy("destination_label"))

eco = spark.table(t(ANALYTICS, "tourism_economy")).filter("baseline_year = 2019 AND comparison_year = 2023")
display(eco.filter("NOT alignment_eligible").groupBy("alignment_exclusion_reason").count())
display(eco.filter("alignment_eligible").groupBy("tourism_economy_pattern").count().orderBy("tourism_economy_pattern"))

In [0]:
# Traveler-segment coverage (AQ1.3): 2023 below-2019 cohort
seg = spark.table(t(ANALYTICS, "traveler_segment_recovery"))
display(seg.agg(F.count("*").alias("cohort_units"),
                F.sum(F.col("domestic_eligible").cast("int")).alias("domestic_eligible"),
                F.sum(F.col("foreign_eligible").cast("int")).alias("foreign_eligible"),
                F.sum(F.col("overseas_filipino_eligible").cast("int")).alias("overseas_filipino_eligible"),
                F.sum(F.coalesce(F.col("foreign_includes_overseas_filipinos"), F.lit(False)).cast("int")).alias("foreign_includes_overseas_filipinos")))

reasons = None
for segment, col in [("Domestic", "domestic_exclusion_reason"), ("Foreign", "foreign_exclusion_reason"),
                     ("Overseas Filipino", "overseas_filipino_exclusion_reason")]:
    part = (seg.filter(F.col(col).isNotNull())
               .groupBy(F.col(col).alias("exclusion_reason")).count()
               .withColumn("segment", F.lit(segment)))
    reasons = part if reasons is None else reasons.unionByName(part)
display(reasons.select("segment", "exclusion_reason", "count").orderBy("segment", F.col("count").desc()))

# Gold caveat fields carried into Analytics
display(eco.filter("derived_values_2019 IS NOT NULL OR derived_values_2023 IS NOT NULL "
                   "OR per_capita_recomputed_2019 = true OR per_capita_recomputed_2023 = true")
           .select("province_name", "derived_values_2019", "derived_values_2023",
                   "per_capita_recomputed_2019", "per_capita_recomputed_2023"))